# Create the dataset for computation

In [1]:
import os
import shutil

HF_CACHE_ROOT = "C:/hf"
os.environ["HF_HOME"] = HF_CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE_ROOT, "d")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_ROOT, "h")
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

old_cache = "C:/Users/Work/.cache/huggingface/datasets/janteichertkluge___amzn_clothes_monthly_diffs_ffill_fixed_splits_buybox/default/0.0.0"
if os.path.exists(old_cache):
    shutil.rmtree(old_cache)  # drop incomplete shards to avoid long-path reuse

import datasets


In [2]:
dataset_cache_root = os.path.join(
    os.environ["HF_DATASETS_CACHE"],
    "janteichertkluge___amzn_clothes_monthly_diffs_ffill_fixed_splits_buybox_extended",
)
if os.path.exists(dataset_cache_root):
    shutil.rmtree(dataset_cache_root, ignore_errors=True)
    print(f"Reset dataset cache at {dataset_cache_root}")
os.makedirs(dataset_cache_root, exist_ok=True)


Reset dataset cache at C:/hf\d\janteichertkluge___amzn_clothes_monthly_diffs_ffill_fixed_splits_buybox_extended


In [3]:
import pandas as pd
import numpy as np

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    
    get_similarities,
    add_lags_and_scale_data,
    compute_neighbors_and_distances,
    compute_neighbor_weighted_prices,
)

In [4]:
txt_only = False
include_embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{include_embeddings}"

In [5]:
from datasets import Dataset

Dataset.cleanup_cache_files

<function datasets.arrow_dataset.Dataset.cleanup_cache_files(self) -> int>

In [6]:
window = 28
mod = 4
max_periods = 54

import os
os.environ["HF_HOME"] = HF_CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE_ROOT, "d")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_ROOT, "h")
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)
os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)
ds_dict = datasets.load_dataset("janteichertkluge/amzn_clothes_monthly_diffs_ffill_fixed_splits_buybox_extended",
                               # download_mode="force_redownload",
                                )



Generating train split:   0%|          | 0/1305284 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1305546 [00:00<?, ? examples/s]

In [7]:
results_256_time_independent, configs_time_independent = load_pred_and_emb(
        embedding_size=256,
        txt_only=txt_only,
        lag_type="time_independent",
        config_path="utils/paths_config.yaml",
        get_lag2_for_diff=False,
    )


results_256_lag1, configs_lag1 = load_pred_and_emb(
        embedding_size=256,
        txt_only=txt_only,
        lag_type="lag1",
        config_path="utils/paths_config.yaml",
        get_lag2_for_diff=False,
    )

if not txt_only:
    results_256_lag2, configs_lag2 = load_pred_and_emb(
            embedding_size=256,
            txt_only=txt_only,
            lag_type="lag1",
            config_path="utils/paths_config.yaml",
            get_lag2_for_diff=False,
        )

[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (14984, 258)
  - train_embeddings_levl:   (14844, 258)
  - val_predictions_levl:    (14984, 4)
  - train_predictions_levl:  (14844, 4)
[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (13750, 258)
  - train_embeddings_levl:   (13618, 258)
  - val_predictions_levl:    (13750, 4)
  - train_predictions_levl:  (13618, 4)
[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (13750, 258)
  - train_embeddings_levl:   (13618, 258)
  - val_predictions_levl:    (13750, 4)
  - train_predictions_levl:  (13618, 4)


In [8]:
window = 28
mod = 4
max_periods = 60


ds_val = ds_dict['test']
ds_train = ds_dict['train']

columns = ['ASIN',
           'SALES_RANK',
           'PRICE',
           'text',
           'date',
           'window',
           'REVIEW_COUNT',
           'RATING',
            'Anzahl Neu Angebote: Aktuell',
            'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
            'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
            'Blitzangebot: Ist Tiefstpreis',
           'subcat_aggregated']

df_val = ds_val.select_columns(columns).to_pandas()
df_val = df_val[df_val["window"] == window].dropna()
df_val["date_t"] = df_val["date"].astype("category").cat.codes
df_val = df_val[df_val["date_t"] <= max_periods]
df_val = df_val[df_val["date_t"] % mod == 0]

df_train = ds_train.select_columns(columns).to_pandas()
df_train = df_train[df_train["window"] == window].dropna()
df_train["date_t"] = df_train["date"].astype("category").cat.codes
df_train = df_train[df_train["date_t"] <= max_periods]
df_train = df_train[df_train["date_t"] % mod == 0]


df_train_val = pd.concat([df_train, df_val], axis=0).reset_index()

# Dataset contains data with different averaged values (by window size).
# Predictions / Embeddings only for one specific window size

num_val = len(df_val)
num_train = len(df_train)

print(f"Number of train samples: {num_train}")
print(f"Number of val samples: {num_val}")

# Create dummy variables for the subcategories
dummy_subcat = pd.get_dummies(df_train_val["subcat_aggregated"]) * 1
dummy_subcat_names = list(dummy_subcat.columns)
dummy_time = pd.get_dummies(df_train_val["date"]) * 1
dummy_time_names = list(dummy_time.columns)

df_train_val = pd.concat([df_train_val, dummy_subcat, dummy_time], axis=1)

df_full = pd.concat([df_train_val, dummy_subcat], axis=1)
df_full.set_index(["ASIN", "date"], inplace=True)

print(f"df_full shape: {df_full.shape}")
print(df_full.columns)

Number of train samples: 14844
Number of val samples: 15007
df_full shape: (29851, 71)
Index(['index', 'SALES_RANK', 'PRICE', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'Anzahl Neu Angebote: Aktuell',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
       'Blitzangebot: Ist Tiefstpreis', 'subcat_aggregated', 'date_t',
       'Active Shorts', 'Blouses & Button-Down Shirts', 'Briefs',
       'Button-Down Shirts', 'Cardigans', 'Casual', 'Cover-Ups',
       'Everyday Bras', 'Fashion Sneakers', 'Jeans', 'Leggings', 'Pullovers',
       'Residual', 'Sets', 'Slippers', 'Sports Bras', 'Sunglasses', 'T-Shirts',
       'Tanks & Camis', 'Tunics', 'Women, Skirts', '2023-01-30', '2023-02-27',
       '2023-03-27', '2023-04-24', '2023-05-22', '2023-06-19', '2023-07-17',
       '2023-08-14', '2023-09-11', '2023-10-09', '2023-11-06', '2023-12-04',
       '2024-01-01', '2024-01-29', '2024-02-26', '2024-03-25', 'Active Shorts',
    

## Create Embeddings, Similarites and PCA based on time independent embeddings

In [9]:
pred_and_emb = results_256_time_independent
embeddings = center_and_norm(pred_and_emb["embeddings"][2], pred_and_emb["embeddings"][0])
print(f"Embedding shape: {embeddings.shape}")
_, cluster_centroids = get_cluster(embeddings, n_clusters=5, n_init=200)
pca_results = get_pca(embeddings, n_components=5)
df_pca = pd.DataFrame(pca_results, columns=[f"pca_{i}" for i in range(pca_results.shape[1])])
df_sim = get_similarities(embeddings, cluster_centroids)

print(f"pca columns: {df_pca.columns}")
print(f"sim columns: {df_sim.columns}")

Embedding shape: (29828, 256)
pca columns: Index(['pca_0', 'pca_1', 'pca_2', 'pca_3', 'pca_4'], dtype='object')
sim columns: Index(['similarity_cluster_0', 'similarity_cluster_1', 'similarity_cluster_2',
       'similarity_cluster_3', 'similarity_cluster_4'],
      dtype='object')


In [10]:
# join to dataframe
df_full = df_full.join(df_pca)
df_full = df_full.join(df_sim)

if include_embeddings:
    df_full = df_full.join(embeddings.add_prefix("emb_"))

In [11]:
df_full.head()

,,index,SALES_RANK,PRICE,text,window,REVIEW_COUNT,RATING,Anzahl Neu Angebote: Aktuell,"Anzahl der abgerufenen Live-Angebote: Neu, FBA","Anzahl der abgerufenen Live-Angebote: Neu, FBM",...,emb_246,emb_247,emb_248,emb_249,emb_250,emb_251,emb_252,emb_253,emb_254,emb_255
ASIN,date,,,,,,,,,,,,,,,,,,,,,
B0006I47DW,2023-01-30,5254,-7.055066,2.209804,Vanity Fair Women's Perfectly Yours Ravissant ...,28.0,11998.428571,4.3,5.0,2.0,4.0,...,0.187661,0.054399,-0.058681,-0.130915,0.05814,0.074891,0.127269,0.079621,-0.050181,-0.122270
B007HQIAL6,2023-01-30,5256,-8.481958,3.351632,NIKE Unisex Performance Cushion Quarter Socks ...,28.0,1317.392857,4.6,21.0,13.0,4.0,...,-0.072339,-0.165929,0.030641,0.000000,0.00000,0.000000,-0.008331,-0.070472,0.153550,0.111725
B00J5Q0N0W,2023-01-30,5257,-7.204919,1.822935,BIRCH's Oval Shoelaces 27 Colors Half Round 1/...,28.0,11708.821429,4.5,1.0,1.0,0.0,...,0.345111,0.107474,-0.105012,0.000000,0.00000,0.000000,0.000000,0.000000,-0.132924,-0.260963
B076GT47DW,2023-01-30,5262,-6.760663,2.756636,Warner's Women's Blissful Benefits Ultrasoft W...,28.0,17187.000000,4.5,3.0,1.0,1.0,...,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
B0788DWB2H,2023-01-30,5263,-3.074675,2.638343,Leggings Depot Women's Relaxed fit Jogger Pant...,28.0,97444.321429,4.4,1.0,1.0,0.0,...,0.047976,0.047958,-0.069324,-0.068678,-0.00125,-0.028660,0.147721,0.049539,-0.055248,-0.003867


In [12]:
df_full_val = df_full.iloc[num_train:, :]
df_full_train = df_full.iloc[:num_train, :]

print(f"df_full shape: {df_full.shape}")
print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")
print(df_full.columns)

df_full shape: (29851, 337)
df_full_train shape: (14844, 337)
df_full_val shape: (15007, 337)
Index(['index', 'SALES_RANK', 'PRICE', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'Anzahl Neu Angebote: Aktuell',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
       ...
       'emb_246', 'emb_247', 'emb_248', 'emb_249', 'emb_250', 'emb_251',
       'emb_252', 'emb_253', 'emb_254', 'emb_255'],
      dtype='object', length=337)


In [13]:
df_full_val = df_full_val.rename(columns={"SALES_RANK": "Q_t", "PRICE": "P_bb_t"})

for i in range(1, 3):
    df_full_val[f"Q_t-{i}"] = df_full_val.groupby("ASIN")["Q_t"].shift(i)
    df_full_val[f"P_bb_t-{i}"] = df_full_val.groupby("ASIN")["P_bb_t"].shift(i)
    df_full_val[f"REVIEW_COUNT_t-{i}"] = df_full_val["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_val[f"RATING_t-{i}"] = df_full_val["RATING"].groupby("ASIN").shift(i)

df_full_val["Delta_Q_t"] = df_full_val["Q_t"] - df_full_val["Q_t-1"]
df_full_val["Delta_P_bb_t"] = df_full_val["P_bb_t"] - df_full_val["P_bb_t-1"]


df_full_train.rename(columns={"SALES_RANK": "Q_t", "PRICE": "P_bb_t"}, inplace=True)
for i in range(1, 3):
    df_full_train[f"Q_t-{i}"] = df_full_train["Q_t"].groupby("ASIN").shift(i)
    df_full_train[f"P_bb_t-{i}"] = df_full_train["P_bb_t"].groupby("ASIN").shift(i)
    df_full_train[f"REVIEW_COUNT_t-{i}"] = df_full_train["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_train[f"RATING_t-{i}"] = df_full_train["RATING"].groupby("ASIN").shift(i)
    
df_full_train["Delta_Q_t"] = df_full_train["Q_t"] - df_full_train["Q_t-1"]
df_full_train["Delta_P_bb_t"] = df_full_train["P_bb_t"] - df_full_train["P_bb_t-1"]


val_predictions_levl = pred_and_emb["predictions"][0]
#val_predictions_diff = pred_and_emb["predictions"][1].rename(columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"})
train_predictions_levl = pred_and_emb["predictions"][2]
#train_predictions_diff = pred_and_emb["predictions"][3].rename(columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"})

df_full_val = df_full_val.join(val_predictions_levl.set_index(["ASIN", "date"]))
df_full_train = df_full_train.join(train_predictions_levl.set_index(["ASIN", "date"]))
#df_full_val = df_full_val.join(val_predictions_diff.set_index(["ASIN", "date"]))
#df_full_train = df_full_train.join(train_predictions_diff.set_index(["ASIN", "date"]))

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")

print(f"df_full_train columns: {df_full_train.columns}")
print(f"df_full_val columns: {df_full_val.columns}")

df_full_train shape: (14844, 349)
df_full_val shape: (15007, 349)
df_full_train columns: Index(['index', 'Q_t', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT', 'RATING',
       'Anzahl Neu Angebote: Aktuell',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
       ...
       'REVIEW_COUNT_t-1', 'RATING_t-1', 'Q_t-2', 'P_bb_t-2',
       'REVIEW_COUNT_t-2', 'RATING_t-2', 'Delta_Q_t', 'Delta_P_bb_t',
       'pred_ml_l', 'pred_ml_m'],
      dtype='object', length=349)
df_full_val columns: Index(['index', 'Q_t', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT', 'RATING',
       'Anzahl Neu Angebote: Aktuell',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
       ...
       'REVIEW_COUNT_t-1', 'RATING_t-1', 'Q_t-2', 'P_bb_t-2',
       'REVIEW_COUNT_t-2', 'RATING_t-2', 'Delta_Q_t', 'Delta_P_bb_t',
       'pred_ml_l', 'pred_ml_m'],
      dtype='object', length=349)


C:\Users\Work\AppData\Roaming\MobaXterm\slash\mx86_64b\var\log\xwin\ipykernel_19960\2076920067.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_full_train.rename(columns={"SALES_RANK": "Q_t", "PRICE": "P_bb_t"}, inplace=True)
C:\Users\Work\AppData\Roaming\MobaXterm\slash\mx86_64b\var\log\xwin\ipykernel_19960\2076920067.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_full_train[f"Q_t-{i}"] = df_full_train["Q_t"].groupby("ASIN").shift(i)
C:\Users\Work\AppData\Roaming\MobaXterm\slash\mx86_64b\var\log\xwin\ipykernel_19960\2076920067.py:16: SettingWithC

### Add Lag 1 predictions to the dataset

In [14]:
val_predictions_levl_lag1 = results_256_lag1["predictions"][0].rename(columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"})
#val_predictions_diff_lag1 = results_256_lag1["predictions"][1].rename(columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"})
train_predictions_levl_lag1 = results_256_lag1["predictions"][2].rename(columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"})
#train_predictions_diff_lag1 = results_256_lag1["predictions"][3].rename(columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"})

df_full_val = df_full_val.join(val_predictions_levl_lag1.set_index(["ASIN", "date"]))
df_full_train = df_full_train.join(train_predictions_levl_lag1.set_index(["ASIN", "date"]))
#df_full_val = df_full_val.join(val_predictions_diff_lag1.set_index(["ASIN", "date"]))
#df_full_train = df_full_train.join(train_predictions_diff_lag1.set_index(["ASIN", "date"]))

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")

print(f"df_full_train columns: {df_full_train.columns}")
print(f"df_full_val columns: {df_full_val.columns}")

df_full_train shape: (14844, 351)
df_full_val shape: (15007, 351)
df_full_train columns: Index(['index', 'Q_t', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT', 'RATING',
       'Anzahl Neu Angebote: Aktuell',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
       ...
       'Q_t-2', 'P_bb_t-2', 'REVIEW_COUNT_t-2', 'RATING_t-2', 'Delta_Q_t',
       'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_lag_1',
       'pred_ml_m_lag_1'],
      dtype='object', length=351)
df_full_val columns: Index(['index', 'Q_t', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT', 'RATING',
       'Anzahl Neu Angebote: Aktuell',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBA',
       'Anzahl der abgerufenen Live-Angebote: Neu, FBM',
       ...
       'Q_t-2', 'P_bb_t-2', 'REVIEW_COUNT_t-2', 'RATING_t-2', 'Delta_Q_t',
       'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_lag_1',
       'pred_ml_m_lag_1'],
      dtype='object', length=351)


write to csv 

In [15]:
df_full_train.to_csv(f"predictions_v2/datasets/{dataframe_name}_train.csv")
df_full_val.to_csv(f"predictions_v2/datasets/{dataframe_name}_val.csv")